# 09 - Export Web Assets (Predictions, Grad-CAM, Model Metadata)

This notebook produces every file the static GitHub Pages site needs, so the
site itself can be plain HTML/CSS/JS with **no backend and no ONNX/TF.js model
conversion** - all inference is precomputed here, once, and saved as JSON +
images.

### Scope: 22 models (not 23)

EfficientNet-B0 was dropped from this project (see
`src/inference/registry.py`'s module docstring): training it crashed the
kernel on this machine (RAM-limited - EfficientNet's MBConv blocks need more
activation memory during training than ResNet18 despite having fewer
parameters), and ResNet18 already had a finalized, test-evaluated checkpoint.
So this notebook exports **21 classical configurations + ResNet18 = 22
models**, all selectable on the site.

### What this notebook produces (all under `../docs/`, so GitHub Pages can
serve directly from a `/docs` folder on the `main` branch - no separate
`gh-pages` branch or build step needed)

1. `docs/assets/images/img_XXX.jpg` - all 715 test images, resized for the
   web (256x256 JPEG).
2. `docs/assets/gradcam/resnet18/img_XXX.jpg` - a Grad-CAM overlay for every
   one of the 715 test images, computed **with respect to ResNet18's actual
   predicted class** for that image (so it always explains "why the model
   reached *this* verdict," whether that verdict is correct or not - this
   is a different choice than `07_error_analysis.ipynb`, which fixed the
   target class to "defective" throughout for a controlled comparison across
   examples; here the goal is explaining each individual prediction, not
   comparing across a fixed axis).
3. `docs/data/images.json` - the 715 images' ids, true labels, and paths.
4. `docs/data/predictions.json` - every (image, model) prediction: label,
   confidence, and (for ResNet18) a path to its Grad-CAM overlay. 22 models x
   715 images = 15,730 entries.
5. `docs/data/models.json` - one entry per model: display name, type,
   feature/classifier or architecture, hyperparameters, and test-set metrics
   (accuracy/precision/recall/F1/ROC AUC) - what the "Model Stats" page reads.

Classical predictions are read directly from
`08_classical_full_benchmark.ipynb`'s already-computed
`../results/predictions/classical_test_predictions.csv` (no need to reload
21 joblib pipelines and recompute). ResNet18 predictions, metrics, and
Grad-CAM are computed fresh in this notebook, since Grad-CAM overlays for all
715 images don't exist yet anywhere.

### Expected runtime

Grad-CAM needs one backward pass per image (can't be batched the way plain
inference can), so the ResNet18 section takes noticeably longer than a normal
inference pass - roughly 10-20 minutes on CPU for all 715 images. Everything
else (classical CSV loading, image copying) is fast.

## 1. Imports

In [1]:
import json
import shutil
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
)

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
except ImportError as exc:
    raise ImportError(
        "pytorch-grad-cam is required for this notebook. Install it with: "
        "pip install grad-cam"
    ) from exc

sys.path.append(str(Path.cwd().parent))

from src.inference.registry import MODEL_REGISTRY

## 2. Paths, Config, and Split (identical to notebooks 02 / 04 / 06 / 07 / 08)

In [2]:
SEED = 42
IMG_SIZE = 224
VAL_RATIO = 0.2
WEB_IMG_SIZE = 256  # display resolution for the site - not the model's input size

DATA_DIR = Path("../data/raw/casting_data/casting_data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")

DOCS_DIR = Path("../docs")
DATA_DIR_OUT = DOCS_DIR / "data"
IMAGES_DIR_OUT = DOCS_DIR / "assets" / "images"
GRADCAM_DIR_OUT = DOCS_DIR / "assets" / "gradcam" / "resnet18"

for d in [DATA_DIR_OUT, IMAGES_DIR_OUT, GRADCAM_DIR_OUT]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CLASS_NAMES = ["def_front", "ok_front"]  # index 0 / 1, matches ImageFolder.class_to_idx

base_train_data = datasets.ImageFolder(TRAIN_DIR)
base_test_data = datasets.ImageFolder(TEST_DIR)

generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(base_train_data), generator=generator).tolist()

test_paths = [Path(p) for p, _ in base_test_data.samples]
test_labels = [label for _, label in base_test_data.samples]

print("Test images:", len(test_paths))

Device: cpu
Test images: 715


## 3. Assign Web IDs to Every Test Image

Each test image gets a short, stable id (`img_000` ... `img_714`) used
everywhere in the exported JSON and as filenames for the copied/generated
images. The id is just the index into `test_paths` (same order everywhere -
`ImageFolder` always lists `def_front/` before `ok_front/`, alphabetically,
so this order is deterministic and reproducible run to run).

In [3]:
def make_image_id(i):
    return f"img_{i:03d}"


image_ids = [make_image_id(i) for i in range(len(test_paths))]
path_to_id = {str(p): image_id for p, image_id in zip(test_paths, image_ids)}

images_meta = [
    {
        "id": image_id,
        "true_label": CLASS_NAMES[label],
        "image": f"assets/images/{image_id}.jpg",
    }
    for image_id, label in zip(image_ids, test_labels)
]

print("Assigned", len(images_meta), "image ids. Example:", images_meta[0])

Assigned 715 image ids. Example: {'id': 'img_000', 'true_label': 'def_front', 'image': 'assets/images/img_000.jpg'}


## 4. Copy and Resize the Test Images for the Web

In [4]:
for path, image_id in tqdm(list(zip(test_paths, image_ids)), desc="Copying images"):
    image_bgr = cv2.imread(str(path))
    if image_bgr is None:
        raise ValueError(f"Could not read image: {path}")

    resized = cv2.resize(image_bgr, (WEB_IMG_SIZE, WEB_IMG_SIZE), interpolation=cv2.INTER_AREA)
    resized_rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

    Image.fromarray(resized_rgb).save(
        IMAGES_DIR_OUT / f"{image_id}.jpg", format="JPEG", quality=85
    )

print("Saved", len(image_ids), "web images to", IMAGES_DIR_OUT.resolve())

Copying images: 100%|██████████| 715/715 [00:12<00:00, 58.49it/s]

Saved 715 web images to D:\Casting Quality Inspection\docs\assets\images


## 5. Classical Model Predictions (Read From Notebook 08's Output)

`08_classical_full_benchmark.ipynb` already computed predictions for all 21
classical configurations on all 715 test images and saved them to
`classical_test_predictions.csv` (long format: one row per image x
configuration). We just read and reshape that here instead of reloading 21
joblib pipelines and re-predicting.

In [5]:
classical_predictions_path = RESULTS_DIR / "predictions" / "classical_test_predictions.csv"
if not classical_predictions_path.exists():
    raise FileNotFoundError(
        f"{classical_predictions_path} not found. Run 08_classical_full_benchmark.ipynb first."
    )

classical_predictions_df = pd.read_csv(classical_predictions_path)
print("Rows:", len(classical_predictions_df), "(expected 21 x 715 = 15,015)")
classical_predictions_df.head()

Rows: 15015 (expected 21 x 715 = 15,015)


,image_path,true_label,feature,classifier,config_id,predicted_label,confidence
0,..\data\raw\casting_data\casting_data\test\def...,0,Raw Pixels,Logistic Regression,raw_pixels_logistic_regression,0,0.999996
1,..\data\raw\casting_data\casting_data\test\def...,0,Raw Pixels,Logistic Regression,raw_pixels_logistic_regression,0,0.999803
2,..\data\raw\casting_data\casting_data\test\def...,0,Raw Pixels,Logistic Regression,raw_pixels_logistic_regression,0,1.000000
3,..\data\raw\casting_data\casting_data\test\def...,0,Raw Pixels,Logistic Regression,raw_pixels_logistic_regression,0,0.999988
4,..\data\raw\casting_data\casting_data\test\def...,0,Raw Pixels,Logistic Regression,raw_pixels_logistic_regression,0,0.999739


In [6]:
predictions = {image_id: {} for image_id in image_ids}

missing_paths = set()

for row in classical_predictions_df.itertuples(index=False):
    image_id = path_to_id.get(row.image_path)
    if image_id is None:
        missing_paths.add(row.image_path)
        continue

    predictions[image_id][row.config_id] = {
        "pred": CLASS_NAMES[int(row.predicted_label)],
        "confidence": round(float(row.confidence), 4),
    }

if missing_paths:
    raise ValueError(
        f"{len(missing_paths)} image path(s) in classical_test_predictions.csv "
        f"did not match any test image from this notebook's split - the two "
        f"notebooks may have been run against different data. Example: "
        f"{next(iter(missing_paths))}"
    )

print("Merged classical predictions for", len(image_ids), "images.")

Merged classical predictions for 715 images.


## 6. ResNet18: Predictions, Metrics, and Grad-CAM (All 715 Test Images)

Unlike the classical models, ResNet18's per-image predictions and Grad-CAM
overlays don't exist yet anywhere, so both are computed here in one pass.
Grad-CAM is computed **with respect to the model's own predicted class** for
each image (not always "defective" as in `07_error_analysis.ipynb`) - the
web demo shows one verdict per image, and the explanation should match that
verdict.

In [7]:
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

resnet18_path = MODELS_DIR / "resnet18_casting.pt"
if not resnet18_path.exists():
    raise FileNotFoundError(
        f"{resnet18_path} not found. Run 06_model_testing.ipynb first."
    )

resnet18 = models.resnet18(weights=None)
resnet18.fc = nn.Linear(resnet18.fc.in_features, 2)
resnet18.load_state_dict(torch.load(resnet18_path, map_location=device))
resnet18 = resnet18.to(device)
resnet18.eval()

target_layers = [resnet18.layer4[-1]]
cam = GradCAM(model=resnet18, target_layers=target_layers)

print("Loaded ResNet18 from", resnet18_path.name)

Loaded ResNet18 from resnet18_casting.pt


In [8]:
def load_for_cam(path):
    pil_image = Image.open(path).convert("RGB")
    tensor = eval_transform(pil_image).unsqueeze(0)

    resized = pil_image.resize((IMG_SIZE, IMG_SIZE))
    rgb_float = np.asarray(resized).astype(np.float32) / 255.0

    return tensor, rgb_float


resnet18_true = []
resnet18_pred = []
resnet18_score = []  # P(def_front), for ROC AUC

for path, image_id, true_label in tqdm(
    list(zip(test_paths, image_ids, test_labels)), desc="ResNet18 + Grad-CAM"
):
    tensor, rgb_float = load_for_cam(path)
    tensor = tensor.to(device)

    with torch.no_grad():
        outputs = resnet18(tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    predicted_class = int(outputs.argmax(dim=1).item())
    p_defective = float(probabilities[0].item())
    confidence = float(probabilities[predicted_class].item())

    resnet18_true.append(true_label)
    resnet18_pred.append(predicted_class)
    resnet18_score.append(p_defective)

    predictions[image_id]["resnet18"] = {
        "pred": CLASS_NAMES[predicted_class],
        "confidence": round(confidence, 4),
        "gradcam": f"assets/gradcam/resnet18/{image_id}.jpg",
    }

    # Grad-CAM w.r.t. the predicted class (whichever class that is)
    targets = [ClassifierOutputTarget(predicted_class)]
    grayscale_cam = cam(input_tensor=tensor, targets=targets)[0]
    overlay = show_cam_on_image(rgb_float, grayscale_cam, use_rgb=True)

    overlay_resized = cv2.resize(overlay, (WEB_IMG_SIZE, WEB_IMG_SIZE), interpolation=cv2.INTER_AREA)
    Image.fromarray(overlay_resized).save(
        GRADCAM_DIR_OUT / f"{image_id}.jpg", format="JPEG", quality=85
    )

print("Done. Saved", len(image_ids), "Grad-CAM overlays to", GRADCAM_DIR_OUT.resolve())

ResNet18 + Grad-CAM: 100%|██████████| 715/715 [02:07<00:00,  5.61it/s]

Done. Saved 715 Grad-CAM overlays to D:\Casting Quality Inspection\docs\assets\gradcam\resnet18


## 7. ResNet18 Test-Set Metrics (for the Model Stats Page)

`precision_score`/`recall_score`/`f1_score` default to treating class **1**
(`ok_front`) as the positive class - that's also what
`08_classical_full_benchmark.ipynb` used (no `pos_label` override there
either), so leaving the default here keeps ResNet18 directly comparable to
all 21 classical configurations on the Model Stats page. ROC AUC is
different: `resnet18_score` is `P(def_front)` (class 0), so `pos_label=0` is
passed explicitly there, the same fix `06_model_testing.ipynb` made - and
this still lands on the correct, standard AUC value, since AUC is invariant
to which class is called "positive" as long as the score is oriented to
match it (unlike precision/recall/F1, which are not symmetric between
classes).

In [9]:
fpr, tpr, _ = roc_curve(resnet18_true, resnet18_score, pos_label=0)

resnet18_metrics = {
    "test_accuracy": accuracy_score(resnet18_true, resnet18_pred),
    "test_precision": precision_score(resnet18_true, resnet18_pred),
    "test_recall": recall_score(resnet18_true, resnet18_pred),
    "test_f1": f1_score(resnet18_true, resnet18_pred),
    "test_roc_auc": auc(fpr, tpr),
}

for key, value in resnet18_metrics.items():
    print(f"{key}: {value:.4f}")

test_accuracy: 0.9986
test_precision: 0.9962
test_recall: 1.0000
test_f1: 0.9981
test_roc_auc: 1.0000


## 8. Assemble `models.json` (Registry + Test Metrics)

In [10]:
classical_metrics_path = RESULTS_DIR / "metrics" / "classical_full_test_results.csv"
if not classical_metrics_path.exists():
    raise FileNotFoundError(
        f"{classical_metrics_path} not found. Run 08_classical_full_benchmark.ipynb first."
    )

classical_metrics_df = pd.read_csv(classical_metrics_path).set_index("config_id")

models_meta = []

for model_id, entry in MODEL_REGISTRY.items():
    meta = dict(entry)
    meta["model_id"] = model_id

    if model_id == "resnet18":
        meta.update({
            "test_accuracy": resnet18_metrics["test_accuracy"],
            "test_precision": resnet18_metrics["test_precision"],
            "test_recall": resnet18_metrics["test_recall"],
            "test_f1": resnet18_metrics["test_f1"],
            "test_roc_auc": resnet18_metrics["test_roc_auc"],
        })
    else:
        row = classical_metrics_df.loc[model_id]
        meta.update({
            "test_accuracy": float(row["test_accuracy"]),
            "test_precision": float(row["test_precision"]),
            "test_recall": float(row["test_recall"]),
            "test_f1": float(row["test_f1"]),
            "test_roc_auc": float(row["test_roc_auc"]),
        })

    models_meta.append(meta)

# Best model first by default - a reasonable initial sort for the site's model picker.
models_meta.sort(key=lambda m: m["test_f1"], reverse=True)

print("Assembled metadata for", len(models_meta), "models.")
pd.DataFrame(models_meta)[["model_id", "display_name", "type", "test_accuracy", "test_f1"]]

Assembled metadata for 22 models.


,model_id,display_name,type,test_accuracy,test_f1
0,resnet18,ResNet18,deep_learning,0.998601,0.998095
1,histogram_svm,Histogram + SVM,classical,0.998601,0.998088
2,histogram_random_forest,Histogram + Random Forest,classical,0.995804,0.994264
3,hog_svm,HOG + SVM,classical,0.993007,0.990548
4,raw_pixels_svm,Raw Pixels + SVM,classical,0.991608,0.988593
5,raw_pixels_random_forest,Raw Pixels + Random Forest,classical,0.984615,0.979206
6,hog_random_forest,HOG + Random Forest,classical,0.983217,0.977099
7,hog_logistic_regression,HOG + Logistic Regression,classical,0.973427,0.964750
8,histogram_logistic_regression,Histogram + Logistic Regression,classical,0.955245,0.937255
9,raw_pixels_logistic_regression,Raw Pixels + Logistic Regression,classical,0.927273,0.903346


## 9. Save Everything

In [11]:
with open(DATA_DIR_OUT / "images.json", "w", encoding="utf-8") as f:
    json.dump(images_meta, f, indent=1, ensure_ascii=False)

with open(DATA_DIR_OUT / "predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=1, ensure_ascii=False)

with open(DATA_DIR_OUT / "models.json", "w", encoding="utf-8") as f:
    json.dump(models_meta, f, indent=1, ensure_ascii=False)

for name in ["images.json", "predictions.json", "models.json"]:
    size_kb = (DATA_DIR_OUT / name).stat().st_size / 1024
    print(f"{name}: {size_kb:.1f} KB")

images.json: 68.9 KB
predictions.json: 1303.6 KB
models.json: 17.1 KB


## 10. Sanity Checks

Before trusting these files in the website, a few structural checks: every
image has exactly 22 predictions, every model_id referenced in
`predictions.json` also exists in `models.json`, and Grad-CAM images only
exist (and are only referenced) for ResNet18.

In [12]:
model_ids_in_meta = {m["model_id"] for m in models_meta}
assert len(model_ids_in_meta) == 22, f"Expected 22 models, found {len(model_ids_in_meta)}"

problems = []
for image_id, model_predictions in predictions.items():
    if set(model_predictions.keys()) != model_ids_in_meta:
        problems.append(image_id)

    if "gradcam" not in model_predictions.get("resnet18", {}):
        problems.append(f"{image_id} missing resnet18 gradcam")

    for model_id, pred in model_predictions.items():
        if model_id != "resnet18" and "gradcam" in pred:
            problems.append(f"{image_id}/{model_id} unexpectedly has a gradcam field")

assert not problems, f"Found {len(problems)} problem(s), e.g. {problems[:5]}"

assert len(list(IMAGES_DIR_OUT.glob("*.jpg"))) == len(image_ids)
assert len(list(GRADCAM_DIR_OUT.glob("*.jpg"))) == len(image_ids)

print("All sanity checks passed:")
print(f"  {len(images_meta)} images x {len(model_ids_in_meta)} models = {len(images_meta) * len(model_ids_in_meta)} predictions")
print(f"  {len(list(IMAGES_DIR_OUT.glob('*.jpg')))} web images saved")
print(f"  {len(list(GRADCAM_DIR_OUT.glob('*.jpg')))} Grad-CAM overlays saved")

All sanity checks passed:
  715 images x 22 models = 15730 predictions
  715 web images saved
  715 Grad-CAM overlays saved


## 11. Summary

`../docs/` now contains everything the static site needs:

- `data/images.json` - 715 test images (id, true label, path).
- `data/predictions.json` - 715 x 22 = 15,730 predictions, each with a label
  and confidence; ResNet18's entries additionally point to a Grad-CAM overlay.
- `data/models.json` - 22 models' metadata and test-set metrics, sorted best
  F1 first.
- `assets/images/*.jpg` - the 715 test images themselves, resized for the web.
- `assets/gradcam/resnet18/*.jpg` - 715 Grad-CAM overlays, one per image,
  explaining ResNet18's actual prediction for that image.

### On repo size

715 images + 715 Grad-CAM overlays at 256x256 JPEG quality 85 should total
roughly 30-60 MB - worth checking with `du -sh ../docs` before committing, and
worth knowing GitHub warns above 50 MB per file (not an issue here, these are
tiny) and recommends keeping whole repos under ~1 GB.

### Next step

Build the actual site (`docs/index.html` + a "Model Stats" page) that reads
these three JSON files and the two image folders - no backend, deployable by
turning on GitHub Pages (`Settings -> Pages -> Deploy from a branch -> main
/docs`) with no further build step. That's a separate deliverable from this
notebook.